# 0. Setup

In [ ]:
# Libraries

# General
import numpy as np                      # imports the Numpy library for numerical tools
import pandas as pd                     # imports the Pandas library for data manipulation and analysis
from scipy import stats

# Plotting Options
from matplotlib import pyplot as plt                           # imports the Pyplot module from the Matplotlib library for plotting
import seaborn as sns 
plt.rcParams['text.usetex'] = True                             # enables LaTeX rendering for text in plots
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'  # LaTeX preamble

# File management
from pathlib import Path                  # imports path tools from Pathlib for working with file paths
import os                               # imports the OS library for interacting with the operating system

def find_project_root(start=Path.cwd(), marker='data'): # defines a function to find the project root directory by looking for a specific marker (default is 'data')
    current = start
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

from tools import text_tools # tools for text processing
from tools import dataframe_tools as dftools # tools for dataframe processing

In [ ]:
biblio = text_tools.bibliography()  # defines a bibliography object
print( biblio.files )               # lists the files in the directory

## 1.1. Vocabulary to word analysis

Here we perform a statiscal analysis and curve fitting for the vocabulary to word distribution

In [ ]:
vocab_to_word = biblio.df_texts['vocabulary']/biblio.df_texts['word_count']

In [ ]:
text_tools.dist_statistics(vocab_to_word)

Possible candidates for this distributions are the distributions: gamma, beta, rayleigh, lognorm and chi2. Therefore we create a dictionary with each distribution and plot them below to visualize

In [ ]:
distributions = {
    "gamma": stats.gamma,
    "beta": stats.beta,
    "rayleigh": stats.rayleigh,
    "lognorm": stats.lognorm,
    "chi2": stats.chi2
}

# eixo comum
x = np.linspace(min(vocab_to_word), max(vocab_to_word), 1000)

# subplots (2 linhas, 3 colunas)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

axes[0].hist(vocab_to_word, bins=10, density=True, alpha=0.7)
axes[0].set_title('Original Histogram')

for i, (name, dist) in enumerate(distributions.items(), start=1):
    ax = axes[i]
    
    # histograma
    ax.hist(vocab_to_word, bins=10, density=True, alpha=0.5)
    
    # fit
    params = dist.fit(vocab_to_word)
    pdf = dist.pdf(x, *params)
    
    # plot
    ax.plot(x, pdf)
    ax.set_title(name)

# ajuste geral
for ax in axes:
    ax.set_xlim(min(vocab_to_word), max(vocab_to_word))

plt.suptitle('Distribution Fits Comparison', fontsize=16)
plt.tight_layout()
plt.show()

Now we perform the fit using the Kolmogorov-Smirnov test

In [ ]:
results = {}

for name, dist in distributions.items():
    # Fit dos parâmetros
    params = dist.fit(vocab_to_word)
    
    # Teste KS
    ks_stat, p_value = stats.kstest(vocab_to_word, name, args=params)
    
    results[name] = {
        "params": params,
        "ks_stat": ks_stat,
        "p_value": p_value
    }

# test cramer von mises

# shapiro wilk (gaussianity)
    

The results are

In [ ]:
# Results 

for name, res in results.items():
    print(f"{name}: p-value = {res['p_value']:.4f}")

Since beta has the highets p-value, we compare it now with what we had

In [ ]:
from scipy.stats import beta

# fit da beta
params = beta.fit(vocab_to_word)

# PDF da beta ajustada
beta_pdf = beta.pdf(x, *params)

# plot
plt.figure(figsize=(10, 6))

# histograma + KDE (seaborn)
sns.histplot(vocab_to_word, bins=10, stat='density', kde=True, alpha=0.4, label='vocab_to_word + KDE')

# beta ajustada
plt.plot(x, beta_pdf, label='Beta fit', linewidth=2, linestyle = 'dashed')

plt.title('Beta Fit vs KDE')
plt.legend()
plt.show()

## 1.2. Vocabulary to sentence analysis

Here we perform a statiscal analysis and curve fitting for the vocabulary to sentence distribution

In [ ]:
vocabulary_to_sentence = biblio.df_texts['vocabulary']/biblio.df_texts['sentence_count']


In [ ]:
# vocabulary to sentence count 

vocabulary_to_sentence = biblio.df_texts['vocabulary']/biblio.df_texts['sentence_count']

plt.hist(vocabulary_to_sentence, density=True, bins= 5)
plt.title('Vocabulary to sentence count ratio distribution') 

In [ ]:
sns.histplot(vocabulary_to_sentence, kde= True, bins=5)
plt.show()

In [ ]:
from scipy.special import gamma as gamma_func  


k = np.linspace(min(vocabulary_to_sentence), max(vocabulary_to_sentence), 1000)
distribution_statistics = dist_statistics(vocabulary_to_sentence)
lambda_poisson = 12
poisson = (lambda_poisson ** k)*(np.exp(-lambda_poisson))/gamma_func(k+1)


In [ ]:

# plot
plt.figure(figsize=(10, 6))

# histograma + KDE (seaborn)
sns.histplot(vocabulary_to_sentence, bins=6, stat='density', kde=True, alpha=0.4, label='vocab_to_word + KDE')

# poisson 

plt.plot(k, poisson)

plt.title('Beta Fit vs KDE')
plt.legend()
plt.show()

Possible candidates to fit this distribution are the Poisson and exponential. Now we do the same process

In [ ]:
distributions = {
    "gamma": stats.gamma,
    "expon": stats.expon,
    'chi2': stats.chi2,
    'beta': stats.beta
}

results = {}

for name, dist in distributions.items():
    # Fit dos parâmetros
    params = dist.fit(vocabulary_to_sentence)
    
    # Teste KS
    ks_stat, p_value = stats.kstest(vocabulary_to_sentence, name, args=params)
    
    results[name] = {
        "params": params,
        "ks_stat": ks_stat,
        "p_value": p_value
    }

for name, res in results.items():
    print(f"{name}: p-value = {res['p_value']:.4f}")

In [ ]:
from scipy.stats import beta
from scipy.stats import gamma

# fit da beta
params_beta = beta.fit(vocabulary_to_sentence)
params_gamma = gamma.fit(vocabulary_to_sentence)

# PDF da beta ajustada
beta_pdf = beta.pdf(k, *params_beta)

# PDF da gamma

gamma_pdf = gamma.pdf(k, *params_gamma)

# plot
plt.figure(figsize=(10, 6))

# histograma + KDE (seaborn)
sns.histplot(vocabulary_to_sentence, bins=20, stat='density', kde=True, alpha=0.4, label='vocab_to_word + KDE')

# beta ajustada
plt.plot(k, beta_pdf, label='Beta fit', linewidth=2, linestyle = 'dashed')
plt.plot(k, gamma_pdf, label = 'Gamma fit', linewidth =2, linestyle = ':')

plt.title('Beta Fit vs KDE')
plt.legend()
plt.show()

### Vocabulary to word distribution for each story 

## 1.1 Individual text analysis 

In this section we make the analysis for each individual text. 

### 1.1.1 The Call of Cthulhu

Using $\tt biblio.sentences$ which a dictionary where the key is the story name and the value is a list with every phrase on the text, we create a new list called $\tt cthulhu\_lens$ with the number of characters of every sentence in the story.

In [ ]:
cthulhu_lens = [ len(phrase) for phrase in biblio.sentences['cthulhu'] ]
#cthulhu_lens_filtered = [length for length in cthulhu_lens if length > 43]
print(cthulhu_lens)

Now we plot the distributions in a histogram. The lenght of characters is in the x-axis and in the y-axis is the number of sentences with that many characters. 

In [ ]:
plt.hist(cthulhu_lens, bins = 20)
plt.xscale('log')
plt.axvline(x=43, linestyle = 'dashed', color = 'black')
plt.title('Distribution of number of characters for each sentence in Call of Cthulhu')

We note that there are a constant piece (the one right before the black-dashed vertical line) that is quite strange. Verifying the minimum of the $\tt cthulhu\_lens$ we note the following

In [ ]:
print(min(cthulhu_lens))

That is, our function that split the text into sentences let some exepctions pass, which are only letters. Thus, we filter this part in order to have a better histogram

In [ ]:
cthulhu_lens = [len(phrase) for phrase in biblio.sentences['cthulhu']]
#Filter out lengths <= 43
cthulhu_lens_filtered = [length for length in cthulhu_lens if length > 43]

plt.hist(cthulhu_lens_filtered, bins=20)
plt.xscale('log') 

Now we can proceed to the curve fitting. One distribution that feels right for this histogram is the beta distribution. 

In [ ]:
# commum axis
x = np.linspace(min(cthulhu_lens), max(cthulhu_lens), 1000)

# subplots 
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
axes = axes.flatten()

axes[0].hist(cthulhu_lens, bins=10, density=True, alpha=0.7)
axes[0].set_title('Original Histogram')

axes[1].hist(cthulhu_lens, bins=10, density=True, alpha=0.5)
axes[1].set_title('Beta distribution for Call of Cthulhu')

    
params_cthulhu = stats.beta.fit(cthulhu_lens)
pdf = stats.beta.pdf(x, *params_cthulhu)
    

axes[1].plot(x, pdf)
axes[1].set_xlim(min(cthulhu_lens), max(cthulhu_lens))

plt.suptitle('Distribution Fits Comparison', fontsize=16)
plt.tight_layout()
plt.show()


Using the function $\tt dist\_statistics$ defined on $\tt text\_tools.py$, we get some relevant statiscal informations regarding the distribution for Call of Cthulhu. 

In [ ]:
text_tools.dist_statistics(cthulhu_lens_filtered)
